# Ders 03 - Temsilci Tasarım Kalıpları

Bu derste, etkili yapay zeka ajanları oluşturmak için üç temel tasarım kalıbını keşfediyoruz:

1. **Net Ajan Talimatları** — Ajan davranışını yönlendiren, rol tanımlayan kesin istemler oluşturma
2. **Pydantic Modellerle Yapılandırılmış Çıktı** — Ajanların öngörülebilir, doğrulanmış veriler döndürmesini sağlama
3. **Tek Sorumluluk Ajanları** — Her biri bir işi iyi yapan odaklanmış ajanlar tasarlama

Her kalıbı bir **seyahat destinasyonu önericisi** senaryosuna uygulayacağız ve kademeli olarak destinasyon öneren, müsaitliği kontrol eden ve lojistiği yöneten bir sistem oluşturacağız.


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Örüntü 1: Net Ajan Talimatları

En etkili örüntü aynı zamanda en basit olanıdır: ajanınız için net, ayrıntılı talimatlar yazmak.

İyi talimatlar şunları tanımlar:
- Ajmanın **kim** olduğu (karakter ve ton)
- Ne yapması gerektiği (adım adım sorumluluklar)
- Nasıl davranması gerektiği (kısıtlamalar ve stil)

Aşağıda, ürettiği her yanıtı şekillendiren açık talimatlarla bir seyahat konsiyerj ajanı oluşturuyoruz.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Desen 2: Pydantic Modeller ile Yapılandırılmış Çıktı

Serbest biçimli metin sohbet için kullanışlıdır, ancak sonraki sistemler yapılandırılmış verilere ihtiyaç duyar.
**Pydantic modellerini** bir **araç fonksiyonu** ile eşleştirerek şunları yapabiliriz:

- Temsilcinin çıktısı için tam bir şema tanımlamak
- Yanıtları otomatik olarak doğrulamak
- Temsilci sonuçlarını uygulama mantığına güvenilir şekilde entegre etmek

Zorlamanın anahtarı, temsilciyi çalıştırırken `response_format` iletmektir. Bu, modelin
serbest biçimli metin yerine doğrulanmış bir `TravelRecommendations` nesnesi
(`response.value` üzerinde mevcut) döndürmesini zorlar. `get_destination_details` aracı da 
yazılı tipte bir `DestinationRecommendation` döndürür, böylece veriler baştan sona yapılandırılmış kalır.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Desen 3: Tek Sorumluluklu Ajanlar

Karmaşık görevler, her biri tek bir sorumluluğa sahip birden çok odaklanmış ajan arasında işin bölünmesinden fayda sağlar:

- Yerler ve müsaitlik hakkında bilgi sahibi bir **Hedef Uzmanı**
- Uçuşlar, oteller ve güzergahlarla ilgilenen bir **Lojistik Planlayıcı**

Bu, her ajanın bağımsız olarak test edilmesini, bakımını ve geliştirilmesini kolaylaştıran *sorumlulukların ayrılması* yazılım mühendisliği ilkesini yansıtır.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Özet

Bu derste, seyahat önerici senaryosuna üç ajan tasarım desenini uyguladık:

| Desen | Temel Fikir | Faydası |
|---|---|---|
| **Net Talimatlar** | Önceden persona, sorumluluklar ve kısıtlamaları tanımla | Tutarlı, marka dostu ajan davranışı |
| **Yapılandırılmış Çıktı** | Yanıt formatı olarak Pydantic modellerini kullan | Doğrulanmış, makine tarafından okunabilir sonuçlar |
| **Tek Sorumluluk** | Her ajana odaklanmış bir görev ver | Test etmesi, bakımı ve bileştirmesi daha kolay |

Bu desenler doğal olarak birleşir — net talimatları yapılandırılmış çıktı ile tek sorumluluk ajanın içinde birleştirerek sağlam, üretime hazır sistemler inşa edebilirsiniz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
